In [13]:
import numpy as np

### Function

In [14]:
def get_frac_active(t_spk_mat, edges, MIN_SPIKES, backbone_threshold):
    """
    Inputs:
    
    t_spk_mat : numpy.ndarray
        Spike matrix of shape (T, N) where T is time bins and N is units
    edges : numpy.ndarray
        Array of shape (B, 2) containing [start, end] indices for each burst
    MIN_SPIKES : int
        Minimum number of spikes required for a unit to be considered active in a burst
    BACKBONE_THRESHOLD : float between 0-1
        Minimum fraction of bursts a unit must be active in to be considered a backbone unit
        
    Returns:

    frac_per_unit : numpy.ndarray
        - 1D array where each value represents a neuron and the fraction of burtsts that involve that neuron. 
        - Example) A value of .75 in the array means Neuron A is active in 75% of bursts

    frac_per_burst : numpy.ndarray
        - 1D array where each value represents a burst and the fraction of neurons that are active 
        in that burst.
        - Example) A value of .75 in the array means Burst A involves 75% of neurons.

    backbone_units : numpy.ndarray
        1D array of the neuron/unit indices that are backbone units. 
    """


    # initiate result array
    spikes_per_burst = np.zeros((t_spk_mat.shape[1], edges.shape[0]))


    # for each unit
    for unit in range(t_spk_mat.shape[1]):

        # obtain spike times in ms
        unit_spk_times = np.where(t_spk_mat[:, unit])[0]

        # for each burst
        for burst in range(edges.shape[0]):

            # obtain all spike times within burst
            burst_times = unit_spk_times[(unit_spk_times >= edges[burst, 0]) & (unit_spk_times <= edges[burst, 1])]

            # store number of spikes in burst
            spikes_per_burst[unit, burst] = len(burst_times)

    # determine bursts above MIN_SPIKES
    above_thresh = spikes_per_burst >= MIN_SPIKES

    # compute fraction of bursts above threshold per unit
    frac_per_unit = np.sum(above_thresh, axis=1) / edges.shape[0]
    frac_per_burst = np.sum(above_thresh, axis=0) / t_spk_mat.shape[1]

    
    backbone_units = np.where(frac_per_unit >= backbone_threshold)[0]

        




    return frac_per_unit, frac_per_burst, backbone_units

### Testing

In [ ]:
t_spk_mat = np.array([
    [0, 0, 0],  # t=0
    [1, 0, 0],  # t=1
    [0, 1, 0],  # t=2
    [1, 0, 1],  # t=3
    [1, 1, 0],  # t=4
    [0, 0, 0],  # t=5
    [0, 1, 1],  # t=6
    [1, 0, 0],  # t=7
    [0, 0, 1],  # t=8
    [0, 1, 0]   # t=9
])
edges = np.array([
    [1, 4],   # First burst from t=1 to t=4
    [6, 9]    # Second burst from t=6 to t=9
])
frac_per_unit, frac_per_burst, backbone_indices = get_frac_active(
    t_spk_mat, edges, 2, 0.5
)

# def test_get_frac_active():
#     try:
#         np.testing.assert_allclose(frac_per_unit, np.array([0.5, 1.0, 0.5]))
#         np.testing.assert_allclose(frac_per_burst, np.array([2/3, 2/3]))
#         np.testing.assert_array_equal(backbone_indices, np.array([0, 1, 2]))

#         return "Basic Functionality Works"
#     except AssertionError as e:
#         return "Test Failed"
    
# test_get_frac_active()

'Test Failed'

In [27]:
def test_get_frac_active():
    # Setup and function call
    #frac_per_unit, frac_per_burst, backbone_indices = get_frac_active(...)
    
    # Collect errors
    errors = []
    
    if not np.allclose(frac_per_unit, np.array([0.5, 1.0, 0.5])):
        errors.append(f"frac_per_unit is incorrect. Expected [0.5, 1.0, 0.5] but got {frac_per_unit}")
    
    if not np.allclose(frac_per_burst, np.array([2/3, 2/3])):
        errors.append(f"frac_per_burst is incorrect. Expected [2/3, 2/3] but got {frac_per_burst}")
    
    if not np.array_equal(backbone_indices, np.array([0, 1, 2])):
        errors.append(f"backbone_indices is incorrect. Expected [0, 1, 2] but got {backbone_indices}")
    
    # Return result
    if errors:
        return "FAILED: " + "; ".join(errors)
    else:
        return "PASSED: All values are correct!"

In [28]:
test_get_frac_active()

'PASSED: All values are correct!'